In [1]:
import os
import re
import unicodedata
import torch
import pandas as pd
import kagglehub
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftConfig, PeftModel

CHECKPOINT_DIR = "/kaggle/input/models/devinshende/byt5sota/pytorch/default/1/kaggle-sota-finetuned-final"
BASE_MODEL_REF = "mattiaangeli/byt5-akkadian-mbr/pyTorch/default"
USE_ADAPTER = False
TEST_PATH = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
SUBMISSION_PATH = "submission.csv"

SRC_LANG = "ar_AR"
TGT_LANG = "en_XX"
MAX_LENGTH = 128
BATCH_SIZE = 32

# Avoid broken SSL env vars and force offline loading
os.environ.pop("SSL_CERT_FILE", None)
os.environ.pop("SSL_CERT_DIR", None)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


In [2]:

def normalize_transliteration(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    replacements = {
        "š": "sh",
        "sz": "sh",
        "ṣ": "s.",
        "s,": "s.",
        "ṭ": "t.",
        "t,": "t.",
        "ḫ": "h.",
        "h,": "h.",
        "ĝ": "g.",
        "g,": "g.",
        "ā": "a",
        "ē": "e",
        "ī": "i",
        "ū": "u",
        "é": "e",
        "í": "i",
        "ú": "u",
        "á": "a",
        "â": "a",
        "à": "a",
        "è": "e",
        "ì": "i",
        "ù": "u",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"\[\.\.\.\]", "...", text)
    text = re.sub(r"\[|\]", "", text)
    text = re.sub(r"\(|\)", "", text)
    text = re.sub(r"\{.*?\}", "", text)
    text = re.sub(r"\^", "", text)
    text = re.sub(r"<{2,}", "<", text)
    text = re.sub(r">{2,}", ">", text)

    text = unicodedata.normalize("NFKC", text)
    text = text.replace(".", "-")
    text = text.replace("-", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_translation(text):
    if not isinstance(text, str):
        return ""

    cleaned = text.strip()

    cleaned = cleaned.replace("…", "...")
    cleaned = re.sub(r"<\s*big_gap\s*>", "...", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"<\s*/\s*big_gap\s*>", "", cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r"\.{4,}", "...", cleaned)
    cleaned = re.sub(r"(\s*\.\.\.\s*)+", " ... ", cleaned)

    cleaned = re.sub(r"\s+", " ", cleaned)
    cleaned = re.sub(r"\s+([,.;:!?])", r"\1", cleaned)
    cleaned = re.sub(r"([(" + "'" + r"])\s+", r"\1", cleaned)
    cleaned = re.sub(r"\s+([)" + "'" + r"])", r"\1", cleaned)

    cleaned = cleaned.strip(" \t\n\r<>")
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    if cleaned == "":
        cleaned = "..."

    return cleaned


def resolve_base_model_source(base_model_name):
    if not isinstance(base_model_name, str) or base_model_name.strip() == "":
        raise ValueError("Invalid base_model_name_or_path in adapter config")

    base_model_name = base_model_name.strip()

    if os.path.isdir(base_model_name):
        return base_model_name

    alt_case_path = base_model_name.replace("/pyTorch/", "/pytorch/")
    if alt_case_path != base_model_name and os.path.isdir(alt_case_path):
        return alt_case_path

    base_override = os.environ.get("BASE_MODEL_DIR")
    if base_override and os.path.isdir(base_override):
        return base_override

    raise FileNotFoundError(
        "Base model directory not found. Expected local path from adapter config: "
        f"{base_model_name}. If this path differs on your machine, set BASE_MODEL_DIR "
        "to the local base-model directory."
    )


def find_local_model_dir(root_dir):
    candidate_dirs = []
    for current_root, _, files in os.walk(root_dir):
        if "config.json" in files and "tokenizer_config.json" in files:
            if "model.safetensors" in files or "pytorch_model.bin" in files:
                candidate_dirs.append(current_root)

    if not candidate_dirs:
        raise FileNotFoundError(f"Could not find a local model directory under: {root_dir}")

    def sort_key(path):
        name = os.path.basename(path)
        if name.startswith("checkpoint-"):
            try:
                return int(name.split("-")[1])
            except Exception:
                return -1
        return -1

    checkpoint_candidates = [p for p in candidate_dirs if os.path.basename(p).startswith("checkpoint-")]
    if checkpoint_candidates:
        return max(checkpoint_candidates, key=sort_key)

    return candidate_dirs[0]


def download_byt5_model(model_ref):
    model_root = kagglehub.model_download(model_ref)
    return find_local_model_dir(model_root)


def load_model_and_tokenizer(checkpoint_dir):
    checkpoint_exists = os.path.isdir(checkpoint_dir)
    use_peft_adapter = USE_ADAPTER and checkpoint_exists and os.path.exists(os.path.join(checkpoint_dir, "adapter_config.json"))

    if use_peft_adapter:
        peft_config = PeftConfig.from_pretrained(checkpoint_dir)
        base_model_name = resolve_base_model_source(peft_config.base_model_name_or_path)
    else:
        base_model_name = download_byt5_model(BASE_MODEL_REF)

    tokenizer_source = checkpoint_dir if use_peft_adapter and os.path.exists(os.path.join(checkpoint_dir, "tokenizer_config.json")) else base_model_name
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source,
        local_files_only=True,
        use_fast=True,
    )

    if hasattr(tokenizer, "src_lang"):
        tokenizer.src_lang = SRC_LANG

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        base_model_name,
        torch_dtype=dtype,
        local_files_only=True,
    )

    model = base_model
    if use_peft_adapter:
        try:
            model = PeftModel.from_pretrained(
                base_model,
                checkpoint_dir,
                local_files_only=True,
            )
        except ValueError as exc:
            if "Target modules" in str(exc) and "not found" in str(exc):
                print("Adapter/base mismatch detected; using base ByT5 model without adapter.")
            else:
                raise

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    return model, tokenizer, device


def batched_generate(model, tokenizer, device, texts):
    predictions = []

    for start in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[start : start + BATCH_SIZE]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            max_length=MAX_LENGTH,
            truncation=True,
            padding=True,
        ).to(device)

        with torch.no_grad():
            generation_kwargs = {
                "max_length": MAX_LENGTH,
                "num_beams": 4,
                "length_penalty": 1.0,
                "repetition_penalty": 2.0,
                "no_repeat_ngram_size": 3,
                "early_stopping": True,
            }

            if hasattr(tokenizer, "lang_code_to_id") and TGT_LANG in tokenizer.lang_code_to_id:
                generation_kwargs["forced_bos_token_id"] = tokenizer.lang_code_to_id[TGT_LANG]

            generated = model.generate(
                **inputs,
                **generation_kwargs,
            )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        decoded = [clean_translation(item) for item in decoded]
        predictions.extend(decoded)

        if (start // BATCH_SIZE) % 10 == 0:
            print(f"Generated {min(start + BATCH_SIZE, len(texts))}/{len(texts)}")

    return predictions


def main():
    print("Loading model and tokenizer...")
    model, tokenizer, device = load_model_and_tokenizer(CHECKPOINT_DIR)
    print(f"Model loaded on {device}")

    print(f"Reading {TEST_PATH}...")
    test_df = pd.read_csv(TEST_PATH)

    if "id" not in test_df.columns:
        raise ValueError("test.csv must contain an 'id' column")
    if "transliteration" not in test_df.columns:
        raise ValueError("test.csv must contain a 'transliteration' column")

    test_df["transliteration"] = test_df["transliteration"].apply(normalize_transliteration)

    print("Running inference...")
    predictions = batched_generate(model, tokenizer, device, test_df["transliteration"].tolist())

    submission = pd.DataFrame({
        "id": test_df["id"],
        "translation": predictions,
    })

    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"Saved {SUBMISSION_PATH} with {len(submission)} rows")


if __name__ == "__main__":
    main()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model and tokenizer...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded on cuda
Reading /kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv...
Running inference...
Generated 4/4
Saved submission.csv with 4 rows


In [3]:
!cat submission.csv

id,translation
0,"I said:'The colony is strict, and I promised to guarantee for judges who became delayed'. Every single shekel of cultivation a"
1,"This day, when I was living in the city-gate of my representa"
2,"As soon as you know that our father is delayed, since"
3,"Our father's colleagues invited me to every single place and wabarrum, or he has given it (to) justice. My dear broth"
